In [2]:
"""
STAP 1 – ClubElo ratings bijwerken
====================================
- Eerste keer: haalt alle data op vanaf 2010-01-01 voor alle Premier League teams
- Volgende keren: gooit de laatste 30 dagen weg en haalt die opnieuw op
  → zo worden placeholder-rijen (elo_to=2026-12-31) altijd vervangen door echte data
- Slaat op naar: C:/Users/semwi/FPL-Core-Insights/data/clubelo_history.csv
"""

import os
import time
import requests
import pandas as pd
from io import StringIO

# ── Configuratie ────────────────────────────────────────────────────────────
ELO_PATH       = r"C:\Users\semwi\FPL-Core-Insights\data\clubelo_history.csv"
ELO_START_DATE  = pd.Timestamp("2010-01-01")
ROLLBACK_DAYS   = 30    # laatste N dagen altijd opnieuw ophalen (vervangt placeholders)
SLEEP_BETWEEN   = 3     # seconden tussen requests (beleefd scrapen)
RETRIES        = 4

# Alle Premier League clubs met hun ClubElo slug + display naam
CLUBELO_TEAMS = {
    "Arsenal":       "Arsenal",
    "AstonVilla":    "Aston Villa",
    "Bournemouth":   "Bournemouth",
    "Brentford":     "Brentford",
    "Brighton":      "Brighton",
    "Chelsea":       "Chelsea",
    "CrystalPalace": "Crystal Palace",
    "Everton":       "Everton",
    "Fulham":        "Fulham",
    "Ipswich":       "Ipswich",
    "Forest":        "Nott'm Forest",
    "Leeds":         "Leeds",
    "Leicester":     "Leicester",
    "Liverpool":     "Liverpool",
    "ManCity":       "Man City",
    "ManUnited":     "Man Utd",
    "Newcastle":     "Newcastle",
    "Southampton":   "Southampton",
    "Tottenham":     "Spurs",
    "WestHam":       "West Ham",
    "Wolves":        "Wolves",
    # Extra clubs die ooit in de PL speelden (zitten al in je CSV)
    "Birmingham":    "Birmingham",
    "Blackburn":     "Blackburn",
    "Blackpool":     "Blackpool",
    "Bolton":        "Bolton",
    "Burnley":       "Burnley",
    "Cardiff":       "Cardiff",
    "Huddersfield":  "Huddersfield",
    "Hull":          "Hull",
    "Luton":         "Luton",
    "Middlesbrough": "Middlesbrough",
    "Norwich":       "Norwich",
    "QPR":           "QPR",
    "Reading":       "Reading",
    "SheffieldUnited": "Sheffield Utd",
    "Stoke":         "Stoke",
    "Sunderland":    "Sunderland",
    "Swansea":       "Swansea",
    "Watford":       "Watford",
    "WestBrom":      "West Brom",
    "Wigan":         "Wigan",
}


# ── Functies ─────────────────────────────────────────────────────────────────

def fetch_clubelo_team(slug: str, from_date: pd.Timestamp, retries: int = RETRIES) -> pd.DataFrame | None:
    """Haalt ELO-history op voor één team van clubelo.com API."""
    url = f"http://api.clubelo.com/{slug}"

    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=30)
            if resp.status_code in (502, 503, 504):
                wait = 2 ** attempt
                print(f"⏳ HTTP {resp.status_code}, wacht {wait}s...", end=" ", flush=True)
                time.sleep(wait)
                continue
            resp.raise_for_status()

            df = pd.read_csv(StringIO(resp.text))

            # Kolommen hernoemen naar onze structuur
            df = df.rename(columns={
                "Club":    "clubelo_club_name",
                "Elo":     "clubelo_rating",
                "From":    "elo_from",
                "To":      "elo_to",
                "Country": "Country",
                "Level":   "Level",
            })

            # Types zetten
            df["elo_from"]       = pd.to_datetime(df["elo_from"], errors="coerce")
            df["elo_to"]         = pd.to_datetime(df["elo_to"],   errors="coerce")
            df["clubelo_rating"] = pd.to_numeric(df["clubelo_rating"], errors="coerce")

            # Verwijder onvolledige rijen
            df = df[df["elo_from"].notna() & df["elo_to"].notna() & df["clubelo_rating"].notna()]

            # Filter op datum
            df = df[df["elo_from"] >= from_date]

            # Voeg onze kolommen toe
            df["team_name"] = CLUBELO_TEAMS[slug]
            df["club_key"]  = slug

            return df[["team_name", "club_key", "elo_from", "elo_to",
                        "clubelo_rating", "clubelo_club_name", "Country", "Level"]]

        except Exception as e:
            wait = 2 ** attempt
            print(f"⏳ Fout ({e}), wacht {wait}s...", end=" ", flush=True)
            time.sleep(wait)

    print(f"⚠️  Mislukt na {retries} pogingen, wordt overgeslagen")
    return None


def update_clubelo(elo_path: str = ELO_PATH) -> pd.DataFrame:
    print("\n══════════════════════════════════════════")
    print("  STAP 1 – ClubElo ratings bijwerken")
    print("══════════════════════════════════════════")

    # ── Bestaande data laden ──────────────────────────────────────────────
    existing = pd.DataFrame()
    is_fresh_start = not os.path.exists(elo_path)

    if not is_fresh_start:
        existing = pd.read_csv(elo_path, parse_dates=["elo_from", "elo_to"])
        print(f"   📂 Bestaande ELO geladen: {len(existing):,} rijen")
        # Laatste 30 dagen altijd opnieuw ophalen → vervangt placeholder-rijen (elo_to=jaar-einde)
        from_date = pd.Timestamp.today().normalize() - pd.Timedelta(days=ROLLBACK_DAYS)
        print(f"   📅 Update modus: laatste {ROLLBACK_DAYS} dagen opnieuw ophalen vanaf {from_date.date()}")
    else:
        from_date = ELO_START_DATE
        print(f"   📅 Eerste keer: ophalen vanaf {from_date.date()} (dit duurt even...)")

    # ── Nieuwe data ophalen ───────────────────────────────────────────────
    frames = []
    total  = len(CLUBELO_TEAMS)

    for i, (slug, display_name) in enumerate(CLUBELO_TEAMS.items(), 1):
        print(f"   [{i:02d}/{total}] 📥 {slug:<18}", end=" ", flush=True)
        df = fetch_clubelo_team(slug, from_date=from_date)
        if df is not None and not df.empty:
            frames.append(df)
            print(f"✅  {len(df)} rijen")
        elif df is not None and df.empty:
            print("⏭️  geen nieuwe rijen")
        else:
            print("❌  overgeslagen")
        time.sleep(SLEEP_BETWEEN)

    if not frames:
        print("\n   ⚠️  Geen nieuwe data opgehaald. Bestaand bestand ongewijzigd.")
        return existing

    new_data = pd.concat(frames, ignore_index=True)
    print(f"\n   📊 Nieuwe rijen opgehaald: {len(new_data):,}")

    # ── Samenvoegen ───────────────────────────────────────────────────────
    if not existing.empty:
        # Verwijder uit bestaand alles vanaf de cutoff (wordt overschreven)
        # Gooi alles weg vanaf rollback-datum (wordt vervangen door verse API data)
        existing_keep = existing[existing["elo_from"] < from_date]
        combined = pd.concat([existing_keep, new_data], ignore_index=True)
        print(f"   🔀 Samengevoegd: {len(existing_keep):,} oud + {len(new_data):,} nieuw")
    else:
        combined = new_data

    # Sorteren en dedupliceren
    combined = (
        combined
        .drop_duplicates(subset=["club_key", "elo_from", "elo_to"], keep="last")
        .sort_values(["club_key", "elo_from"])
        .reset_index(drop=True)
    )

    # ── Opslaan ───────────────────────────────────────────────────────────
    os.makedirs(os.path.dirname(elo_path), exist_ok=True)
    combined.to_csv(elo_path, index=False)

    print(f"\n   ✅ Klaar! {len(combined):,} rijen opgeslagen → {elo_path}")
    print("══════════════════════════════════════════\n")
    return combined


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = update_clubelo()
    print(df.tail(5).to_string(index=False))


══════════════════════════════════════════
  STAP 1 – ClubElo ratings bijwerken
══════════════════════════════════════════
   📂 Bestaande ELO geladen: 58,951 rijen
   📅 Update modus: laatste 30 dagen opnieuw ophalen vanaf 2026-02-02
   [01/41] 📥 Arsenal            ✅  12 rijen
   [02/41] 📥 AstonVilla         ✅  17 rijen
   [03/41] 📥 Bournemouth        ✅  15 rijen
   [04/41] 📥 Brentford          ✅  12 rijen
   [05/41] 📥 Brighton           ✅  16 rijen
   [06/41] 📥 Chelsea            ✅  17 rijen
   [07/41] 📥 CrystalPalace      ✅  17 rijen
   [08/41] 📥 Everton            ✅  15 rijen
   [09/41] 📥 Fulham             ✅  15 rijen
   [10/41] 📥 Ipswich            ✅  16 rijen
   [11/41] 📥 Forest             ✅  19 rijen
   [12/41] 📥 Leeds              ✅  14 rijen
   [13/41] 📥 Leicester          ✅  11 rijen
   [14/41] 📥 Liverpool          ✅  13 rijen
   [15/41] 📥 ManCity            ✅  14 rijen
   [16/41] 📥 ManUnited          ✅  12 rijen
   [17/41] 📥 Newcastle          ✅  12 rijen
   [18/41] 📥 South

In [7]:
"""
STAP 2 – Sofascore match + player data bijwerken (2025-2026)
=============================================================
- Haalt alle Premier League wedstrijden op via de Sofascore API
- Voegt alleen nieuwe wedstrijden toe die nog niet in de CSV staan
- Schrijft naar:
    match_path:  C:/Users/semwi/FPL-Core-Insights/data/Seasonal data/matches/2025-2026_raw.csv
    player_path: C:/Users/semwi/FPL-Core-Insights/data/Seasonal data/players/2025-2026_players.csv
"""

import os
import csv
import time
import requests

# ── Configuratie ─────────────────────────────────────────────────────────────
API_KEY     = "2b277abcd2msh0e5627048810020p119057jsn4412b05235c9"
SEASON_ID   = 76986          # Sofascore season ID voor 2025-2026 Premier League
TOURNAMENT  = 17             # Premier League tournament ID
SEASON_STR  = "2025-2026"

MATCH_PATH  = r"C:\Users\semwi\FPL-Core-Insights\data\Seasonal data\matches\2025-2026_raw.csv"
PLAYER_PATH = r"C:\Users\semwi\FPL-Core-Insights\data\Seasonal data\players\2025-2026_players.csv"

DELAY   = 0.3   # seconden tussen requests
RETRIES = 3

HEADERS = {
    "x-rapidapi-key":  API_KEY,
    "x-rapidapi-host": "sofascore.p.rapidapi.com"
}

# ── Hulpfuncties ──────────────────────────────────────────────────────────────

def api_get(url: str, params: dict = None) -> dict | None:
    """GET request met retry logica."""
    for attempt in range(RETRIES):
        try:
            resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
            data = resp.json()
            if "message" in data:
                print(f"\n  ⛔ QUOTA OP: {data['message']}")
                return None
            return data
        except Exception as e:
            wait = 2 ** attempt
            print(f"  ⏳ Fout ({e}), wacht {wait}s...", end=" ", flush=True)
            time.sleep(wait)
    return None


def load_done_ids(path: str) -> set:
    """Laad alle match_ids die al in de CSV staan."""
    done = set()
    if not os.path.exists(path):
        return done
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            mid = row.get("match_id", "")
            if mid:
                done.add(int(mid))
    return done


def fetch_all_event_ids() -> list[dict]:
    """Haal alle wedstrijd-IDs op via tournaments/get-matches (gepagineerd)."""
    print("   📋 Wedstrijd-IDs ophalen...")
    all_events = []
    page = 0

    while True:
        url = (f"https://sofascore.p.rapidapi.com/tournaments/get-matches"
               f"?tournamentId={TOURNAMENT}&seasonId={SEASON_ID}&pageIndex={page}")
        data = api_get(url)
        if data is None:
            print(f"\n   ⛔ Gestopt bij pagina {page} (quota of fout)")
            break

        events = data.get("events", [])
        if not events:
            print(f"   ✅ Alle paginas verwerkt ({page} paginas totaal)")
            break

        for e in events:
            all_events.append({
                "event_id": e["id"],
                "round":    e.get("roundInfo", {}).get("round"),
                "status":   e.get("status", {}).get("description", ""),
            })

        print(f"   Pagina {page:02d}: {len(events)} wedstrijden")
        page += 1
        time.sleep(DELAY)

    print(f"   📊 Totaal gevonden: {len(all_events)} wedstrijden")
    return all_events


def fetch_match_details(event_id: int) -> tuple[dict, list]:
    """Haalt details, statistieken en lineups op voor één wedstrijd.
    Geeft (match_row dict, lineup_rows list) terug."""

    # 1. Match details
    data = api_get(f"https://sofascore.p.rapidapi.com/matches/detail?matchId={event_id}")
    time.sleep(DELAY)
    ed = data.get("event", {}) if data else {}

    # 2. Statistieken
    data = api_get(f"https://sofascore.p.rapidapi.com/matches/get-statistics?matchId={event_id}")
    time.sleep(DELAY)
    stats_raw = data.get("statistics", []) if data else []

    stats_flat = {}
    for period in stats_raw:
        if period.get("period") != "ALL":
            continue
        for group in period.get("groups", []):
            for item in group.get("statisticsItems", []):
                key = item.get("key", "")
                stats_flat[f"home_{key}"] = item.get("homeValue")
                stats_flat[f"away_{key}"] = item.get("awayValue")

    match_row = {
        "match_id":     event_id,
        "season":       SEASON_STR,
        "round":        ed.get("roundInfo", {}).get("round"),
        "timestamp":    ed.get("startTimestamp"),
        "status":       ed.get("status", {}).get("description"),
        "home_team":    ed.get("homeTeam", {}).get("name"),
        "away_team":    ed.get("awayTeam", {}).get("name"),
        "home_team_id": ed.get("homeTeam", {}).get("id"),
        "away_team_id": ed.get("awayTeam", {}).get("id"),
        "home_goals":   ed.get("homeScore", {}).get("current"),
        "away_goals":   ed.get("awayScore", {}).get("current"),
        "venue":        ed.get("venue", {}).get("name") if ed.get("venue") else None,
        "referee":      ed.get("referee", {}).get("name") if ed.get("referee") else None,
        "attendance":   ed.get("attendance"),
        **stats_flat
    }

    # 3. Lineups
    data = api_get(f"https://sofascore.p.rapidapi.com/matches/get-lineups?matchId={event_id}")
    time.sleep(DELAY)
    lineups_raw = data if data else {}

    home_formation = lineups_raw.get("home", {}).get("formation")
    away_formation = lineups_raw.get("away", {}).get("formation")

    lineup_rows = []
    for side in ["home", "away"]:
        team_data  = lineups_raw.get(side, {})
        formation  = home_formation if side == "home" else away_formation
        for p in team_data.get("players", []):
            player = p.get("player", {})
            stats  = p.get("statistics", {})
            lineup_rows.append({
                "match_id":            event_id,
                "season":              SEASON_STR,
                "home_team":           ed.get("homeTeam", {}).get("name"),
                "away_team":           ed.get("awayTeam", {}).get("name"),
                "home_goals":          ed.get("homeScore", {}).get("current"),
                "away_goals":          ed.get("awayScore", {}).get("current"),
                "round":               ed.get("roundInfo", {}).get("round"),
                "timestamp":           ed.get("startTimestamp"),
                "side":                side,
                "formation":           formation,
                "player_id":           player.get("id"),
                "player_name":         player.get("name"),
                "short_name":          player.get("shortName"),
                "position":            p.get("position"),
                "shirt_number":        p.get("shirtNumber"),
                "substitute":          p.get("substitute"),
                "captain":             p.get("captain"),
                "nationality":         player.get("country", {}).get("name"),
                "height":              player.get("height"),
                "market_value":        player.get("proposedMarketValueRaw", {}).get("value") if player.get("proposedMarketValueRaw") else None,
                "rating":              stats.get("rating"),
                "minutes_played":      stats.get("minutesPlayed"),
                "touches":             stats.get("touches"),
                "total_pass":          stats.get("totalPass"),
                "accurate_pass":       stats.get("accuratePass"),
                "total_long_balls":    stats.get("totalLongBalls"),
                "accurate_long_balls": stats.get("accurateLongBalls"),
                "total_cross":         stats.get("totalCross"),
                "accurate_cross":      stats.get("accurateCross"),
                "key_pass":            stats.get("keyPass"),
                "total_shots":         stats.get("totalShots"),
                "on_target":           stats.get("onTargetScoringAttempt"),
                "shot_off_target":     stats.get("shotOffTarget"),
                "blocked_shot":        stats.get("blockedScoringAttempt"),
                "goals":               stats.get("goals"),
                "goal_assist":         stats.get("goalAssist"),
                "big_chance_created":  stats.get("bigChanceCreated"),
                "big_chance_missed":   stats.get("bigChanceMissed"),
                "hit_woodwork":        stats.get("hitWoodwork"),
                "duel_won":            stats.get("duelWon"),
                "duel_lost":           stats.get("duelLost"),
                "aerial_won":          stats.get("aerialWon"),
                "aerial_lost":         stats.get("aerialLost"),
                "total_tackle":        stats.get("totalTackle"),
                "won_tackle":          stats.get("wonTackle"),
                "interception_won":    stats.get("interceptionWon"),
                "total_clearance":     stats.get("totalClearance"),
                "outfielder_block":    stats.get("outfielderBlock"),
                "total_contest":       stats.get("totalContest"),
                "won_contest":         stats.get("wonContest"),
                "dispossessed":        stats.get("dispossessed"),
                "possession_lost":     stats.get("possessionLostCtrl"),
                "unsuccessful_touch":  stats.get("unsuccessfulTouch"),
                "ball_recovery":       stats.get("ballRecovery"),
                "was_fouled":          stats.get("wasFouled"),
                "fouls":               stats.get("fouls"),
                "total_offside":       stats.get("totalOffside"),
                "penalty_won":         stats.get("penaltyWon"),
                "penalty_conceded":    stats.get("penaltyConceded"),
                "penalty_miss":        stats.get("penaltyMiss"),
                "error_led_to_goal":   stats.get("errorLeadToGoal"),
                "saves":               stats.get("saves"),
                "saves_inside_box":    stats.get("savedShotsFromInsideTheBox"),
                "penalty_save":        stats.get("penaltySave"),
                "punches":             stats.get("punches"),
                "acc_own_half_pass":   stats.get("accurateOwnHalfPasses"),
                "acc_opp_half_pass":   stats.get("accurateOppositionHalfPasses"),
            })

    return match_row, lineup_rows


def write_match(match_row: dict, path: str):
    fixed_cols = ["match_id", "season", "round", "timestamp", "status",
                  "home_team", "away_team", "home_team_id", "away_team_id",
                  "home_goals", "away_goals", "venue", "referee", "attendance"]
    stat_cols  = sorted(k for k in match_row if k not in fixed_cols)
    all_cols   = fixed_cols + stat_cols

    file_exists = os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=all_cols, extrasaction="ignore")
        if not file_exists:
            writer.writeheader()
        writer.writerow(match_row)


def write_players(lineup_rows: list, path: str):
    if not lineup_rows:
        return
    file_exists = os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=lineup_rows[0].keys(), extrasaction="ignore")
        if not file_exists:
            writer.writeheader()
        writer.writerows(lineup_rows)


# ── Main ──────────────────────────────────────────────────────────────────────

def update_sofascore():
    print("\n══════════════════════════════════════════")
    print("  STAP 2 – Sofascore data bijwerken")
    print(f"  Seizoen: {SEASON_STR}")
    print("══════════════════════════════════════════")

    # Welke match_ids staan al in de CSV?
    done_ids = load_done_ids(MATCH_PATH)
    print(f"   📂 Al verwerkt: {len(done_ids)} wedstrijden")

    # Haal alle event IDs op voor dit seizoen
    all_events = fetch_all_event_ids()

    # Filter: alleen afgeronde wedstrijden die nog niet in de CSV staan
    to_process = [
        e for e in all_events
        if e["event_id"] not in done_ids
        and e["status"] in ("Ended", "finished", "After Penalties", "After Extra Time")
    ]

    print(f"\n   🆕 Nieuwe wedstrijden te verwerken: {len(to_process)}")

    if not to_process:
        print("   ✅ Alles al up-to-date!")
        return

    request_count = 0
    for i, event in enumerate(to_process, 1):
        event_id = event["event_id"]
        rnd      = event["round"]
        print(f"   [{i:02d}/{len(to_process)}] GW{rnd:02d} – event {event_id}...", end=" ", flush=True)

        match_row, lineup_rows = fetch_match_details(event_id)
        request_count += 3  # details + stats + lineups

        home = match_row.get("home_team", "?")
        away = match_row.get("away_team", "?")
        hg   = match_row.get("home_goals", "?")
        ag   = match_row.get("away_goals", "?")

        write_match(match_row, MATCH_PATH)
        write_players(lineup_rows, PLAYER_PATH)

        print(f"✅  {home} {hg}-{ag} {away} | {len(lineup_rows)} spelers | requests: {request_count}")

    print(f"\n   ✅ Klaar! {request_count} requests gebruikt")
    print(f"   📁 Matches:  {MATCH_PATH}")
    print(f"   📁 Spelers:  {PLAYER_PATH}")
    print("══════════════════════════════════════════\n")


if __name__ == "__main__":
    update_sofascore()


══════════════════════════════════════════
  STAP 2 – Sofascore data bijwerken
  Seizoen: 2025-2026
══════════════════════════════════════════
   📂 Al verwerkt: 271 wedstrijden
   📋 Wedstrijd-IDs ophalen...
   Pagina 00: 30 wedstrijden
   Pagina 01: 30 wedstrijden
   Pagina 02: 30 wedstrijden
   Pagina 03: 30 wedstrijden
   Pagina 04: 30 wedstrijden
   Pagina 05: 30 wedstrijden
   Pagina 06: 30 wedstrijden
   Pagina 07: 30 wedstrijden
   Pagina 08: 30 wedstrijden
   Pagina 09: 15 wedstrijden
   ✅ Alle paginas verwerkt (10 paginas totaal)
   📊 Totaal gevonden: 285 wedstrijden

   🆕 Nieuwe wedstrijden te verwerken: 14
   [01/14] GW28 – event 14023979... ✅  Wolverhampton 2-0 Aston Villa | 40 spelers | requests: 3
   [02/14] GW28 – event 14023970... ✅  Bournemouth 1-1 Sunderland | 40 spelers | requests: 6
   [03/14] GW28 – event 14023975... ✅  Burnley 3-4 Brentford | 40 spelers | requests: 9
   [04/14] GW28 – event 14023977... ✅  Newcastle United 2-3 Everton | 40 spelers | requests: 12
  